# Объединение откалиброванных dark images для использования в последующих шагах редукции



Заключительный шаг - объединить отдельные откалиброванные dark images в одно
объединенное изображение. Это объединенное изображение будет иметь меньше шума, чем отдельные
изображения, минимизируя шум, добавляемый к остальным изображениям при вычитании dark.

Независимо от того, какой путь вы выбрали при калибровке biases (с
overscan или без), должна быть папка с именем либо `example1-reduced`, либо
`example2-reduced`, содержащая откалиброванные bias и dark images. Если ее
нет, пожалуйста, запустите предыдущий notebook перед продолжением.

In [ ]:
from pathlib import Path
import os

from astropy.nddata import CCDData
from astropy.stats import mad_std

import ccdproc as ccdp
import matplotlib.pyplot as plt
import numpy as np

from convenience_functions import show_image

In [ ]:
# Use custom style for larger fonts and figures
plt.style.use('guide.mplstyle')

## Рекомендуемые настройки для объединения изображений

Как обсуждалось в [notebook об объединении изображений](01-06-Image-combination.ipynb), рекомендуется
объединять, усредняя отдельные изображения, но с sigma clip для удаления
экстремальных значений.

[ccdproc](https://ccdproc.readthedocs.org) предоставляет два способа объединения:

+ Объектно-ориентированный интерфейс, построенный вокруг объекта `Combiner`, описанный в
[документации ccdproc по объединению изображений](https://ccdproc.readthedocs.io/en/latest/image_combination.html).
+ Функция [`combine`](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.combine.html#ccdproc.combine), которую мы будем использовать здесь, поскольку функция
позволяет указать максимальный объем памяти, который следует использовать во время
объединения. Эта функция может быть существенной в зависимости от того, сколько изображений вам нужно
объединить, насколько они велики и сколько памяти у вашего компьютера.

*ПРИМЕЧАНИЕ: Если используется версия ccdproc ниже 2.0, установите ограничение памяти в
2-3 раза меньше, чем вы хотите, чтобы было максимальное потребление памяти.*

## Example 1: Криогенно охлаждаемая камера

Оставшаяся часть этого раздела предполагает, что откалиброванные bias images находятся в
папке `example1-reduced`, которая была создана в предыдущем notebook.

In [ ]:
calibrated_path = Path('example1-reduced')
reduced_images = ccdp.ImageFileCollection(calibrated_path)

### Создание объединенного изображения для каждого времени экспозиции в Example 1

В этом наборе данных есть несколько времен экспозиции dark. Преобразуя времена
в summary table в set, возвращаются только уникальные значения.

In [ ]:
darks = reduced_images.summary['imagetyp'] == 'DARK'
dark_times = set(reduced_images.summary['exptime'][darks])
print(dark_times)

Приведенный ниже код проходит по временам экспозиции dark и для каждого времени экспозиции:

+ выбирает соответствующие откалиброванные dark images,
+ объединяет их с помощью функции `combine`,
+ добавляет ключевое слово `COMBINED` в header, чтобы последующие шаги калибровки могли
легко идентифицировать, какой bias использовать, и
+ записывает файл, имя которого включает время экспозиции.

In [ ]:
for exp_time in sorted(dark_times):
    calibrated_darks = reduced_images.files_filtered(imagetyp='dark', exptime=exp_time,
                                                     include_path=True)

    combined_dark = ccdp.combine(calibrated_darks,
                                 method='average',
                                 sigma_clip=True, sigma_clip_low_thresh=5, sigma_clip_high_thresh=5,
                                 sigma_clip_func=np.ma.median, sigma_clip_dev_func=mad_std,
                                 mem_limit=350e6
                                )

    combined_dark.meta['combined'] = True

    dark_file_name = 'combined_dark_{:6.3f}.fit'.format(exp_time)
    combined_dark.write(calibrated_path / dark_file_name)

### Результат для Example 1

Ниже показаны одно откалиброванное 300-секундное dark image и объединенное 300-секундное изображение.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

show_image(CCDData.read(calibrated_darks[0]).data, cmap='gray', ax=ax1, fig=fig)
ax1.set_title('Single calibrated dark')
show_image(combined_dark.data, cmap='gray', ax=ax2, fig=fig)
ax2.set_title('{} dark images combined'.format(len(calibrated_darks)))

## Example 2: Термоэлектрически охлаждаемая камера

Процесс объединения изображений точно такой же, как в example 1. Единственное
отличие - это директория, содержащая откалиброванные bias frames.

In [ ]:
calibrated_path = Path('example2-reduced')
reduced_images = ccdp.ImageFileCollection(calibrated_path)

### Создание объединенного изображения для каждого времени экспозиции в Example 2

В этом примере есть только darks с одним временем экспозиции.

In [ ]:
darks = reduced_images.summary['imagetyp'] == 'DARK'
dark_times = set(reduced_images.summary['exptime'][darks])
print(dark_times)

Несмотря на то, что есть только одно время экспозиции, мы можем с таким же успехом повторно использовать
код сверху.

In [ ]:
for exp_time in sorted(dark_times):
    calibrated_darks = reduced_images.files_filtered(imagetyp='dark', exptime=exp_time,
                                                     include_path=True)

    combined_dark = ccdp.combine(calibrated_darks,
                                 method='average',
                                 sigma_clip=True, sigma_clip_low_thresh=5, sigma_clip_high_thresh=5,
                                 sigma_clip_func=np.ma.median, signma_clip_dev_func=mad_std,
                                 mem_limit=350e6
                                )

    combined_dark.meta['combined'] = True

    dark_file_name = 'combined_dark_{:6.3f}.fit'.format(exp_time)
    combined_dark.write(calibrated_path / dark_file_name)

### Результат для Example 2

Разница между одним откалиброванным bias image и объединенным bias
image гораздо более очевидна в этом случае.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

show_image(CCDData.read(calibrated_darks[0]).data, cmap='gray', ax=ax1, fig=fig)
ax1.set_title('Single calibrated dark')
show_image(combined_dark.data, cmap='gray', ax=ax2, fig=fig)
ax2.set_title('{} dark images combined'.format(len(calibrated_darks)))